# PhasePhyto Data Downloader to Google Drive

Use this notebook once to download and prepare plant disease datasets into Google Drive. The training notebook can then reuse the saved folders without downloading again.

Supported datasets:

- **PlantVillage** (source domain, ~1.5 GB via Kaggle API)
- **PlantDoc** (target domain, ~1 GB via GitHub)
- **Cassava Leaf Disease** (additional target, ~5.5 GB via Kaggle competitions API). Requires one-time competition-rules acceptance at <https://www.kaggle.com/c/cassava-leaf-disease-classification> before the API will return a zip. Staged here for future cross-benchmark OOD evaluation; not wired into the training notebook yet.

**Output layout:**

```text
/content/drive/MyDrive/PhasePhyto/data/plant_disease/
  plantvillage/
    <class_name>/image...
  plantdoc/
    train/<class_name>/image...
    test/<class_name>/image...
  cassava/
    <class_name>/image...           # reorganized from flat Kaggle layout
  dataset_manifest.json
```

After this notebook finishes, open `PhasePhyto_Colab.ipynb` and set:

```python
CONFIG["use_synthetic"] = False
CONFIG["plantvillage_dir"] = "/content/drive/MyDrive/PhasePhyto/data/plant_disease/plantvillage"
CONFIG["plantdoc_dir"] = "/content/drive/MyDrive/PhasePhyto/data/plant_disease/plantdoc"
# Optional (not yet consumed by training loop):
# CONFIG["cassava_dir"] = "/content/drive/MyDrive/PhasePhyto/data/plant_disease/cassava"
```

The training notebook will automatically resolve `plantdoc/test/<class>` for target evaluation.


> For the strict apple-overlap workflow (PlantVillage + PlantDoc + Plant Pathology 2021) with tar-backed Drive storage and SSD hydration, use `notebooks/PhasePhyto_Apple_Overlap_Colab.ipynb`.


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

CONFIG = {
    # Google Drive destination
    "drive_project_dir": "/content/drive/MyDrive/PhasePhyto",
    "dataset_subdir": "data/plant_disease",
    "archive_subdir": "data/archives",

    # What to download
    "download_plantvillage": True,
    "download_plantdoc": True,
    # Cassava is off by default: it is ~5.5 GB and requires accepting the
    # competition rules in-browser before the Kaggle API will return the zip.
    # Flip to True only after you have clicked "Join Competition" at
    # https://www.kaggle.com/c/cassava-leaf-disease-classification
    "download_cassava": False,

    # Set True only when you intentionally want to overwrite existing data.
    "force_redownload": False,

    # Create reusable tar archives in Drive. Copying/extracting one tar per
    # dataset is much faster in Colab than rsyncing thousands of small files.
    "create_tar_archives": True,
    "force_recreate_archives": False,

    # PlantVillage Kaggle dataset slug
    "plantvillage_kaggle_dataset": "abdallahalidev/plantvillage-dataset",

    # Cassava Leaf Disease Kaggle competition slug
    "cassava_kaggle_competition": "cassava-leaf-disease-classification",

    # Kaggle credential options. If kaggle_json_drive_path exists, it is copied.
    # Otherwise the notebook asks you to upload kaggle.json interactively.
    "kaggle_json_drive_path": "/content/drive/MyDrive/kaggle.json",

    # Local temporary working directory. Colab local disk is faster than Drive
    # for zip extraction and git clone; final curated data is copied to Drive.
    "tmp_dir": "/content/phasephyto_data_tmp",
}

for k, v in CONFIG.items():
    print(f"{k}: {v}")


In [ ]:
# ============================================================
# MOUNT DRIVE + PREPARE DIRECTORIES
# ============================================================

from pathlib import Path
import csv
import json
import os
import re
import shutil
import subprocess
import sys
from datetime import datetime

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_PROJECT_DIR = Path(CONFIG["drive_project_dir"])
DATASET_ROOT = DRIVE_PROJECT_DIR / CONFIG["dataset_subdir"]
ARCHIVE_DIR = DRIVE_PROJECT_DIR / CONFIG["archive_subdir"]
TMP_DIR = Path(CONFIG["tmp_dir"])

PLANTVILLAGE_DIR = DATASET_ROOT / "plantvillage"
PLANTDOC_DIR = DATASET_ROOT / "plantdoc"
CASSAVA_DIR = DATASET_ROOT / "cassava"
MANIFEST_PATH = DATASET_ROOT / "dataset_manifest.json"

for directory in [DRIVE_PROJECT_DIR, DATASET_ROOT, ARCHIVE_DIR, TMP_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Drive project dir: {DRIVE_PROJECT_DIR}")
print(f"Dataset root:      {DATASET_ROOT}")
print(f"Archive dir:       {ARCHIVE_DIR}")
print(f"Temporary dir:     {TMP_DIR}")


In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def run(cmd):
    """Run a shell command and fail immediately on errors."""
    print("\n$ " + " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True)


def safe_rmtree(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)


def image_count(root):
    root = Path(root)
    if not root.exists():
        return 0
    return sum(
        1 for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )


def class_counts(root):
    """Count direct images under immediate class directories."""
    root = Path(root)
    counts = {}
    if not root.exists():
        return counts
    for class_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        n = sum(
            1 for p in class_dir.iterdir()
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        )
        if n:
            counts[class_dir.name] = n
    return counts


def normalize_class_name(name):
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def find_image_folder_candidate(root):
    """Find a folder whose immediate children are class dirs containing images."""
    root = Path(root)
    candidates = []
    for p in root.rglob("*"):
        if not p.is_dir():
            continue
        counts = class_counts(p)
        if counts:
            candidates.append((len(counts), image_count(p), p))
    if not candidates:
        return None
    candidates.sort(key=lambda item: (item[0], item[1]), reverse=True)
    return candidates[0][2]


def copy_class_folders(src_root, dest_root):
    src_root = Path(src_root)
    dest_root = Path(dest_root)
    dest_root.mkdir(parents=True, exist_ok=True)
    copied = 0
    for class_dir in sorted(p for p in src_root.iterdir() if p.is_dir()):
        if class_counts(src_root).get(class_dir.name, 0) == 0:
            continue
        shutil.copytree(class_dir, dest_root / class_dir.name, dirs_exist_ok=True)
        copied += 1
    return copied


def print_counts(label, root):
    counts = class_counts(root)
    print(f"{label}: {sum(counts.values())} images across {len(counts)} direct classes at {root}")
    return counts


def cassava_disease_to_class_name(disease):
    """Normalize a Cassava competition disease string to `Cassava___<Disease>`.

    Examples:
      "Cassava Bacterial Blight (CBB)" -> "Cassava___Bacterial_Blight"
      "Healthy"                        -> "Cassava___healthy"
    """
    text = (disease or "").strip()
    if "(" in text:
        text = text.split("(", 1)[0].strip()
    if text.lower().startswith("cassava "):
        text = text[len("cassava "):].strip()
    if text.lower() == "healthy":
        return "Cassava___healthy"
    slug = "_".join(text.split())
    return f"Cassava___{slug}"


# Fallback Cassava label map, used only if the shipped
# label_num_to_disease_map.json is missing or corrupt.
CASSAVA_LABEL_TO_CLASS = {
    "0": "Cassava___Bacterial_Blight",
    "1": "Cassava___Brown_Streak_Disease",
    "2": "Cassava___Green_Mottle",
    "3": "Cassava___Mosaic_Disease",
    "4": "Cassava___healthy",
}


PLANTDOC_TO_PLANTVILLAGE = {
    "Apple Scab Leaf": "Apple___Apple_scab",
    "Apple leaf": "Apple___healthy",
    "Apple rust leaf": "Apple___Cedar_apple_rust",
    "Bell_pepper leaf": "Pepper,_bell___healthy",
    "Bell_pepper leaf spot": "Pepper,_bell___Bacterial_spot",
    "Blueberry leaf": "Blueberry___healthy",
    "Cherry leaf": "Cherry_(including_sour)___healthy",
    "Corn Gray leaf spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight",
    "Corn rust leaf": "Corn_(maize)___Common_rust_",
    "Grape leaf": "Grape___healthy",
    "grape leaf": "Grape___healthy",
    "Grape leaf black rot": "Grape___Black_rot",
    "grape leaf black rot": "Grape___Black_rot",
    "Peach leaf": "Peach___healthy",
    "Potato leaf early blight": "Potato___Early_blight",
    "Potato leaf late blight": "Potato___Late_blight",
    "Raspberry leaf": "Raspberry___healthy",
    "Soyabean leaf": "Soybean___healthy",
    "Soybean leaf": "Soybean___healthy",
    "Squash Powdery mildew leaf": "Squash___Powdery_mildew",
    "Strawberry leaf": "Strawberry___healthy",
    "Strawberry leaf scorch": "Strawberry___Leaf_scorch",
    "Tomato Early blight leaf": "Tomato___Early_blight",
    "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "Tomato leaf": "Tomato___healthy",
    "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "Tomato leaf late blight": "Tomato___Late_blight",
    "Tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
    "Tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato mold leaf": "Tomato___Leaf_Mold",
    "Tomato two spotted spider mites leaf": "Tomato___Spider_mites Two-spotted_spider_mite",
}


In [ ]:
# ============================================================
# SET UP KAGGLE CREDENTIALS (used by PlantVillage and Cassava)
# ============================================================

needs_kaggle = CONFIG.get("download_plantvillage") or CONFIG.get("download_cassava")

if needs_kaggle:
    run([sys.executable, "-m", "pip", "install", "-q", "kaggle"])

    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json = kaggle_dir / "kaggle.json"
    drive_kaggle_json = Path(CONFIG["kaggle_json_drive_path"])

    if not kaggle_json.exists():
        if drive_kaggle_json.exists():
            shutil.copy2(drive_kaggle_json, kaggle_json)
            print(f"Copied Kaggle credentials from {drive_kaggle_json}")
        else:
            print("Upload your kaggle.json now. Get it from Kaggle > Account > Create New API Token.")
            from google.colab import files
            uploaded = files.upload()
            if "kaggle.json" not in uploaded:
                raise RuntimeError("kaggle.json was not uploaded; cannot use Kaggle API.")
            shutil.move("kaggle.json", kaggle_json)
            # Save a reusable copy to Drive for future sessions.
            shutil.copy2(kaggle_json, drive_kaggle_json)
            print(f"Saved reusable Kaggle credentials to {drive_kaggle_json}")

    os.chmod(kaggle_json, 0o600)
    print(f"Kaggle credentials ready: {kaggle_json}")
else:
    print("Skipping Kaggle setup (no Kaggle-backed datasets selected).")


In [ ]:
# ============================================================
# DOWNLOAD + PREPARE PLANTVILLAGE INTO DRIVE
# ============================================================

if CONFIG["download_plantvillage"]:
    if PLANTVILLAGE_DIR.exists() and image_count(PLANTVILLAGE_DIR) > 0 and not CONFIG["force_redownload"]:
        print(f"PlantVillage already exists at {PLANTVILLAGE_DIR}; skipping download.")
    else:
        if CONFIG["force_redownload"]:
            safe_rmtree(PLANTVILLAGE_DIR)
        tmp_pv = TMP_DIR / "plantvillage"
        safe_rmtree(tmp_pv)
        tmp_pv.mkdir(parents=True, exist_ok=True)

        run([
            "kaggle", "datasets", "download",
            "-d", CONFIG["plantvillage_kaggle_dataset"],
            "-p", tmp_pv,
        ])

        zip_files = list(tmp_pv.glob("*.zip"))
        if not zip_files:
            raise RuntimeError(f"No zip file downloaded to {tmp_pv}")
        run(["unzip", "-q", zip_files[0], "-d", tmp_pv / "extracted"])

        # Prefer the color image folder when available.
        color_candidates = [p for p in (tmp_pv / "extracted").rglob("color") if p.is_dir()]
        src = color_candidates[0] if color_candidates else find_image_folder_candidate(tmp_pv / "extracted")
        if src is None:
            raise RuntimeError("Could not find PlantVillage image-folder layout after extraction.")

        copied = copy_class_folders(src, PLANTVILLAGE_DIR)
        print(f"Copied {copied} PlantVillage class folders from {src} to {PLANTVILLAGE_DIR}")

    pv_counts = print_counts("PlantVillage", PLANTVILLAGE_DIR)
else:
    print("Skipping PlantVillage download.")
    pv_counts = print_counts("PlantVillage existing", PLANTVILLAGE_DIR)


In [ ]:
# ============================================================
# DOWNLOAD + PREPARE PLANTDOC INTO DRIVE
# ============================================================

if CONFIG["download_plantdoc"]:
    if PLANTDOC_DIR.exists() and image_count(PLANTDOC_DIR) > 0 and not CONFIG["force_redownload"]:
        print(f"PlantDoc already exists at {PLANTDOC_DIR}; skipping download.")
    else:
        if CONFIG["force_redownload"]:
            safe_rmtree(PLANTDOC_DIR)
        tmp_pd = TMP_DIR / "plantdoc_repo"
        safe_rmtree(tmp_pd)

        run([
            "git", "clone", "--depth", "1",
            "https://github.com/pratikkayal/PlantDoc-Dataset.git",
            tmp_pd,
        ])

        PLANTDOC_DIR.mkdir(parents=True, exist_ok=True)
        split_copied = False
        for split in ["train", "test"]:
            src_split = tmp_pd / split
            if src_split.exists() and class_counts(src_split):
                dest_split = PLANTDOC_DIR / split
                if CONFIG["force_redownload"]:
                    safe_rmtree(dest_split)
                copied = copy_class_folders(src_split, dest_split)
                print(f"Copied PlantDoc {split}: {copied} class folders")
                split_copied = True

        if not split_copied:
            src = find_image_folder_candidate(tmp_pd)
            if src is None:
                raise RuntimeError("Could not find PlantDoc image-folder layout after clone.")
            copied = copy_class_folders(src, PLANTDOC_DIR)
            print(f"Copied PlantDoc flat layout: {copied} class folders from {src}")

    pd_train_counts = print_counts("PlantDoc train", PLANTDOC_DIR / "train")
    pd_test_counts = print_counts("PlantDoc test", PLANTDOC_DIR / "test")
    if not pd_test_counts:
        pd_test_counts = print_counts("PlantDoc flat", PLANTDOC_DIR)
else:
    print("Skipping PlantDoc download.")
    pd_train_counts = print_counts("PlantDoc train existing", PLANTDOC_DIR / "train")
    pd_test_counts = print_counts("PlantDoc test existing", PLANTDOC_DIR / "test")


In [ ]:
# ============================================================
# CREATE REUSABLE TAR ARCHIVES IN DRIVE
# ============================================================

archive_paths = {}

if CONFIG["create_tar_archives"]:
    ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
    for dataset_name, dataset_dir in [
        ("plantvillage", PLANTVILLAGE_DIR),
        ("plantdoc", PLANTDOC_DIR),
        ("cassava", CASSAVA_DIR),
    ]:
        archive_path = ARCHIVE_DIR / f"{dataset_name}.tar"
        archive_paths[dataset_name] = str(archive_path)

        if not dataset_dir.exists() or image_count(dataset_dir) == 0:
            print(f"Skipping archive for {dataset_name}: no images found at {dataset_dir}")
            continue

        if archive_path.exists() and not CONFIG["force_recreate_archives"]:
            size_gb = archive_path.stat().st_size / 1e9
            print(f"Archive already exists: {archive_path} ({size_gb:.2f} GB); skipping")
            continue

        if archive_path.exists():
            archive_path.unlink()

        # Archive with top-level folder name matching dataset_name, so the
        # training notebook can extract directly into /content/data.
        run(["tar", "-cf", archive_path, "-C", DATASET_ROOT, dataset_name])
        size_gb = archive_path.stat().st_size / 1e9
        print(f"Created archive: {archive_path} ({size_gb:.2f} GB)")
else:
    print("Skipping tar archive creation because create_tar_archives=False")


In [ ]:
# ============================================================
# AUDIT CLASS OVERLAP + WRITE DATASET MANIFEST
# ============================================================

source_norm = {normalize_class_name(name): name for name in pv_counts}
target_source = pd_test_counts or pd_train_counts
target_norm = {normalize_class_name(name): name for name in target_source}
common = sorted(set(source_norm).intersection(target_norm))

overlap = [
    {
        "normalized": key,
        "source": source_norm[key],
        "target": target_norm[key],
        "source_images": pv_counts[source_norm[key]],
        "target_images": target_source[target_norm[key]],
    }
    for key in common
]

mapped_overlap = []
for target_name in sorted(target_source):
    source_name = PLANTDOC_TO_PLANTVILLAGE.get(target_name)
    if source_name is None or source_name not in pv_counts:
        continue
    mapped_overlap.append({
        "source": source_name,
        "target": target_name,
        "source_images": pv_counts[source_name],
        "target_images": target_source[target_name],
    })

# Cassava uses disjoint species-specific classes (Cassava___*), so there is
# no expected overlap with PlantVillage's per-crop class names. Report the
# per-class counts separately.
cassava_class_rows = [
    {"class": name, "num_images": count}
    for name, count in sorted(cassava_counts.items())
]

manifest = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "dataset_root": str(DATASET_ROOT),
    "plantvillage_dir": str(PLANTVILLAGE_DIR),
    "plantdoc_dir": str(PLANTDOC_DIR),
    "plantdoc_train_dir": str(PLANTDOC_DIR / "train"),
    "plantdoc_test_dir": str(PLANTDOC_DIR / "test"),
    "cassava_dir": str(CASSAVA_DIR),
    "archive_dir": str(ARCHIVE_DIR),
    "archives": archive_paths,
    "plantvillage_num_classes": len(pv_counts),
    "plantvillage_num_images": sum(pv_counts.values()),
    "plantdoc_train_num_classes": len(pd_train_counts),
    "plantdoc_train_num_images": sum(pd_train_counts.values()),
    "plantdoc_test_num_classes": len(pd_test_counts),
    "plantdoc_test_num_images": sum(pd_test_counts.values()),
    "cassava_num_classes": len(cassava_counts),
    "cassava_num_images": sum(cassava_counts.values()),
    "cassava_classes": cassava_class_rows,
    "normalized_overlap_num_classes": len(overlap),
    "normalized_overlap": overlap,
    "mapped_overlap_num_classes": len(mapped_overlap),
    "mapped_overlap": mapped_overlap,
    "source_only_classes": sorted(source_norm[k] for k in set(source_norm) - set(target_norm)),
    "target_only_classes": sorted(target_norm[k] for k in set(target_norm) - set(source_norm)),
}

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps({
    "dataset_root": manifest["dataset_root"],
    "plantvillage": [manifest["plantvillage_num_classes"], manifest["plantvillage_num_images"]],
    "plantdoc_train": [manifest["plantdoc_train_num_classes"], manifest["plantdoc_train_num_images"]],
    "plantdoc_test": [manifest["plantdoc_test_num_classes"], manifest["plantdoc_test_num_images"]],
    "cassava": [manifest["cassava_num_classes"], manifest["cassava_num_images"]],
    "normalized_overlap_num_classes": manifest["normalized_overlap_num_classes"],
    "mapped_overlap_num_classes": manifest["mapped_overlap_num_classes"],
}, indent=2))
print(f"\nManifest saved to: {MANIFEST_PATH}")


In [ ]:
# ============================================================
# PATHS TO USE LATER IN THE TRAINING NOTEBOOK
# ============================================================

print("Copy these into notebooks/PhasePhyto_Colab.ipynb CONFIG:")
print("\nCONFIG['use_synthetic'] = False")
print(f"CONFIG['plantvillage_dir'] = '{PLANTVILLAGE_DIR}'")
print(f"CONFIG['plantdoc_dir'] = '{PLANTDOC_DIR}'")
if image_count(CASSAVA_DIR) > 0:
    print(f"# Optional (not yet consumed by the training loop):")
    print(f"# CONFIG['cassava_dir'] = '{CASSAVA_DIR}'")
print("CONFIG['storage_backend'] = 'drive'")
print(f"CONFIG['drive_project_dir'] = '{DRIVE_PROJECT_DIR}'")
print("CONFIG['hydrate_local_data_from_archives'] = True")
print(f"CONFIG['drive_archive_dir'] = '{ARCHIVE_DIR}'")

print("\nIf you want faster training after downloading, you can copy/sync data from Drive to /content/data at runtime, or point the training notebook to a mounted SSD.")


In [ ]:
# ============================================================
# PATHS TO USE LATER IN THE TRAINING NOTEBOOK
# ============================================================

print("Copy these into notebooks/PhasePhyto_Colab.ipynb CONFIG:")
print("\nCONFIG['use_synthetic'] = False")
print(f"CONFIG['plantvillage_dir'] = '{PLANTVILLAGE_DIR}'")
print(f"CONFIG['plantdoc_dir'] = '{PLANTDOC_DIR}'")
print("CONFIG['storage_backend'] = 'drive'")
print(f"CONFIG['drive_project_dir'] = '{DRIVE_PROJECT_DIR}'")
print("CONFIG['hydrate_local_data_from_archives'] = True")
print(f"CONFIG['drive_archive_dir'] = '{ARCHIVE_DIR}'")

print("\nIf you want faster training after downloading, you can copy/sync data from Drive to /content/data at runtime, or point the training notebook to a mounted SSD.")
